### Traffic Prediction using GRU + Fuzzy Logic + PSO

1. GRU: Learns temporal patterns from traffic data
2. Fuzzy Logic with Membership Functions: Adjusts predictions with gradual transitions
3. PSO: Optimizes GRU hyperparameters

In [1]:
!pip install pyswarms

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import pyswarms as ps
from tqdm import tqdm
import warnings
import pickle

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

In [3]:
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [4]:
# Load the traffic data
filepath = "traffic.csv"
df_full = pd.read_csv(filepath)
print(f"dataset shape: {df_full.shape}")

dataset shape: (48120, 4)


In [5]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48120 entries, 0 to 48119
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DateTime  48120 non-null  object
 1   Junction  48120 non-null  int64 
 2   Vehicles  48120 non-null  int64 
 3   ID        48120 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 1.5+ MB


In [6]:
# parse datetime
df_full['DateTime'] = pd.to_datetime(df_full['DateTime'], errors='coerce')
df_full = df_full.dropna(subset=['DateTime'])

# GRU Model Class Definition
class TrafficGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super(TrafficGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [7]:
# Fuzzy Membership Functions
def membership_night(hour):
    if 0 <= hour <= 3:
        return 1.0
    elif 3 < hour <= 6:
        return (6 - hour) / 3
    elif 22 <= hour <= 24:
        return (hour - 22) / 2
    else:
        return 0.0

def membership_peak(hour):
    if 6 <= hour < 7:
        return (hour - 6)
    elif 7 <= hour <= 8:
        return 1.0
    elif 8 < hour <= 9:
        return (9 - hour)
    elif 16 <= hour < 17:
        return (hour - 16)
    elif 17 <= hour <= 18:
        return 1.0
    elif 18 < hour <= 19:
        return (19 - hour)
    else:
        return 0.0

In [8]:
# Function to preprocess data for a junction
def preprocess_junction(df, junction_id):
    print(f"Junction {junction_id}")
    print(f"{'-'*60}")
    
    # Filter to specific junction
    df = df[df['Junction'] == junction_id].reset_index(drop=True)
    print(f"Total records: {len(df)}")
    
    # Extract time features
    df['Hour'] = df['DateTime'].dt.hour
    df['Weekday'] = df['DateTime'].dt.weekday
    
    # Cyclical encoding
    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    
    # Flags
    df['Is_weekend'] = (df['Weekday'] >= 5).astype(int)
    df['Is_night'] = ((df['Hour'] >= 0) & (df['Hour'] <= 5)).astype(int)
    df['Is_peak'] = (((df['Hour'] >= 7) & (df['Hour'] <= 9)) | 
                     ((df['Hour'] >= 17) & (df['Hour'] <= 19))).astype(int)
    
    # Lag features
    df['Lag_1'] = df['Vehicles'].shift(1)
    df['Lag_24'] = df['Vehicles'].shift(24)
    df['Roll_mean_24'] = df['Vehicles'].rolling(window=24, min_periods=1).mean()
    
    # Drop NaN
    df = df.dropna().reset_index(drop=True)
    print(f"After preprocessing: {len(df)} records")
    
    return df

In [9]:
# Function to create sequences
def create_sequences(X_scaled, y_scaled, sequence_length=24):
    X_sequences = []
    y_sequences = []
    
    for i in range(sequence_length, len(X_scaled)):
        X_sequences.append(X_scaled[i-sequence_length:i])
        y_sequences.append(y_scaled[i])
    
    return np.array(X_sequences), np.array(y_sequences)

In [10]:
# Function to train model with PSO
def train_with_pso(X_train, y_train, X_val, y_val, use_pso=True):
    input_size = X_train.shape[2]
    
    if use_pso:
        print("\nPSO Optimization...")
        
        def evaluate_hyperparameters(params):
            n_particles = params.shape[0]
            losses = []
            
            for i in range(n_particles):
                hidden_size = int(np.clip(params[i, 0], 16, 128))
                num_layers = int(np.clip(params[i, 1], 1, 3))
                learning_rate = float(np.clip(params[i, 2], 0.0001, 0.01))
                
                try:
                    model = TrafficGRU(input_size, hidden_size, num_layers, dropout=0.2)
                    criterion = nn.MSELoss()
                    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
                    
                    X_train_t = torch.FloatTensor(X_train)
                    y_train_t = torch.FloatTensor(y_train)
                    X_val_t = torch.FloatTensor(X_val)
                    y_val_t = torch.FloatTensor(y_val)
                    
                    model.train()
                    for epoch in range(10):
                        optimizer.zero_grad()
                        outputs = model(X_train_t)
                        loss = criterion(outputs, y_train_t)
                        loss.backward()
                        optimizer.step()
                    
                    model.eval()
                    with torch.no_grad():
                        val_outputs = model(X_val_t)
                        val_loss = criterion(val_outputs, y_val_t).item()
                    
                    losses.append(val_loss)
                except:
                    losses.append(1e6)
            
            return np.array(losses)
        
        # PSO settings
        n_particles = 8
        n_iterations = 5
        min_bounds = np.array([16, 1, 0.0001])
        max_bounds = np.array([128, 3, 0.01])
        bounds = (min_bounds, max_bounds)
        options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
        
        optimizer_pso = ps.single.GlobalBestPSO(
            n_particles=n_particles,
            dimensions=3,
            options=options,
            bounds=bounds
        )
        
        cost, pos = optimizer_pso.optimize(evaluate_hyperparameters, iters=n_iterations)
        
        best_hidden_size = int(pos[0])
        best_num_layers = int(pos[1])
        best_learning_rate = pos[2]
        
        print(f"Best hidden_size: {best_hidden_size}")
        print(f"Best num_layers: {best_num_layers}")
        print(f"Best learning_rate: {best_learning_rate:.6f}")
    else:
        best_hidden_size = 64
        best_num_layers = 2
        best_learning_rate = 0.001
    
    return best_hidden_size, best_num_layers, best_learning_rate, input_size

In [11]:
# Function to train final model
def train_final_model(X_train, y_train, best_hidden_size, best_num_layers, 
                      best_learning_rate, input_size, epochs=50):
    print(f"\nTraining Final GRU Model...")
    
    model = TrafficGRU(
        input_size=input_size,
        hidden_size=best_hidden_size,
        num_layers=best_num_layers,
        dropout=0.2
    )
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=best_learning_rate)
    
    # 80-20 validation split
    val_split = int(len(X_train) * 0.8)
    X_train_part = X_train[:val_split]
    y_train_part = y_train[:val_split]
    X_val_part = X_train[val_split:]
    y_val_part = y_train[val_split:]
    
    X_train_tensor = torch.FloatTensor(X_train_part)
    y_train_tensor = torch.FloatTensor(y_train_part)
    X_val_tensor = torch.FloatTensor(X_val_part)
    y_val_tensor = torch.FloatTensor(y_val_part)
    
    train_losses = []
    val_losses = []
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor)
            val_losses.append(val_loss.item())
        model.train()
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train: {loss.item():.6f}, Val: {val_loss.item():.6f}')
    
    print(f"Training improvement: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")
    
    return model, train_losses, val_losses

In [12]:
# Main training loop for all junctions
JUNCTIONS = [1, 2, 3, 4]
SEQUENCE_LENGTH = 24
USE_PSO = True
EPOCHS = 50

# Store results
all_results = []

for junction_id in JUNCTIONS:
    print(f"# Training Junction {junction_id}")
    print(f"{'-'*60}\n")
    
    # 1. Preprocess data
    df = preprocess_junction(df_full.copy(), junction_id)
    
    # 2. Feature engineering
    feature_cols = ['Hour_sin', 'Hour_cos', 'Lag_1', 'Lag_24', 'Roll_mean_24']
    target_col = 'Vehicles'
    
    X_data = df[feature_cols].values
    y_data = df[target_col].values.reshape(-1, 1)
    
    # 3. Scale data
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_scaled = scaler_X.fit_transform(X_data)
    y_scaled = scaler_y.fit_transform(y_data)
    
    # 4. Create sequences
    X_seq, y_seq = create_sequences(X_scaled, y_scaled, SEQUENCE_LENGTH)
    print(f"Sequences created: {X_seq.shape}")
    
    # 5. Train/Val/Test split
    train_size = int(len(X_seq) * 0.70)
    val_size = int(len(X_seq) * 0.15)
    
    X_train = X_seq[:train_size]
    X_val = X_seq[train_size:train_size + val_size]
    X_test = X_seq[train_size + val_size:]
    
    y_train = y_seq[:train_size]
    y_val = y_seq[train_size:train_size + val_size]
    y_test = y_seq[train_size + val_size:]
    
    # 6. PSO Optimization
    best_hidden_size, best_num_layers, best_learning_rate, input_size = \
        train_with_pso(X_train, y_train, X_val, y_val, USE_PSO)
    
    # 7. Train final model
    model, train_losses, val_losses = train_final_model(
        X_train, y_train, best_hidden_size, best_num_layers, 
        best_learning_rate, input_size, EPOCHS
    )
    
    # 8. Evaluate on test set
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.FloatTensor(X_test)
        y_pred_scaled = model(X_test_tensor).numpy()
    
    y_pred_gru = scaler_y.inverse_transform(y_pred_scaled)
    y_test_actual = scaler_y.inverse_transform(y_test)
    
    # 9. Apply fuzzy logic
    test_start_idx = train_size + val_size + SEQUENCE_LENGTH
    hours_test = df['Hour'].iloc[test_start_idx:test_start_idx + len(y_test_actual)].values
    is_weekend_test = df['Is_weekend'].iloc[test_start_idx:test_start_idx + len(y_test_actual)].values
    
    y_pred_fuzzy = []
    for i in range(len(y_pred_gru)):
        pred = y_pred_gru[i][0]
        hour = hours_test[i]
        is_weekend = is_weekend_test[i]
        
        night_degree = membership_night(hour)
        peak_degree = membership_peak(hour)
        adjustment = 0.0
        
        if is_weekend == 0:
            adjustment -= 0.15 * night_degree
            adjustment += 0.10 * peak_degree
        if is_weekend == 1:
            adjustment -= 0.08
        
        adjusted_pred = max(0, pred * (1 + adjustment))
        y_pred_fuzzy.append(adjusted_pred)
    
    y_pred_fuzzy = np.array(y_pred_fuzzy).reshape(-1, 1)
    
    # 10. Calculate metrics
    mse_gru = mean_squared_error(y_test_actual, y_pred_gru)
    rmse_gru = np.sqrt(mse_gru)
    mae_gru = mean_absolute_error(y_test_actual, y_pred_gru)
    r2_gru = r2_score(y_test_actual, y_pred_gru)
    
    mse_fuzzy = mean_squared_error(y_test_actual, y_pred_fuzzy)
    rmse_fuzzy = np.sqrt(mse_fuzzy)
    mae_fuzzy = mean_absolute_error(y_test_actual, y_pred_fuzzy)
    r2_fuzzy = r2_score(y_test_actual, y_pred_fuzzy)
    
    print(f"\nResults for Junction {junction_id}:")
    print(f"GRU RMSE: {rmse_gru:.4f} | GRU+Fuzzy RMSE: {rmse_fuzzy:.4f}")
    print(f"GRU R²: {r2_gru:.4f} | GRU+Fuzzy R²: {r2_fuzzy:.4f}")
    
    # 11. SAVE MODEL AND SCALERS
    print(f"\nSave model for Junction {junction_id}...")
    
    torch.save({
        'model_state_dict': model.state_dict(),
        'hidden_size': best_hidden_size,
        'num_layers': best_num_layers,
        'input_size': input_size,
        'best_params': {
            'hidden_size': best_hidden_size,
            'num_layers': best_num_layers,
            'learning_rate': best_learning_rate
        },
        'metrics': {
            'rmse_gru': rmse_gru,
            'rmse_fuzzy': rmse_fuzzy,
            'r2_gru': r2_gru,
            'r2_fuzzy': r2_fuzzy
        }
    }, f'models/gru_model_junction_{junction_id}.pth')
    
    with open(f'models/scaler_X_junction_{junction_id}.pkl', 'wb') as f:
        pickle.dump(scaler_X, f)
    
    with open(f'models/scaler_y_junction_{junction_id}.pkl', 'wb') as f:
        pickle.dump(scaler_y, f)
    
    print(f"Model saved: models/gru_model_junction_{junction_id}.pth")

    # Store results
    all_results.append({
        'Junction': junction_id,
        'RMSE_GRU': rmse_gru,
        'RMSE_Fuzzy': rmse_fuzzy,
        'R2_GRU': r2_gru,
        'R2_Fuzzy': r2_fuzzy,
        'Hidden_Size': best_hidden_size,
        'Num_Layers': best_num_layers,
        'Learning_Rate': best_learning_rate
    })

# Summary table
print("SUMMARY: ALL JUNCTIONS")
print(f"{'-'*80}\n")

results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

# Save summary
results_df.to_csv('results/training_summary.csv', index=False)
print(f"\nSummary saved to results/training_summary.csv")
print(f"\nAll models trained and saved successfully!")


# Training Junction 1
------------------------------------------------------------

Junction 1
------------------------------------------------------------
Total records: 14592
After preprocessing: 14568 records
Sequences created: (14544, 24, 5)

PSO Optimization...


2026-02-10 23:32:49,735 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0.00603
2026-02-10 23:36:44,261 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.006034339312463999, best pos: [5.60164293e+01 1.64572355e+00 6.75809806e-03]


Best hidden_size: 56
Best num_layers: 1
Best learning_rate: 0.006758

Training Final GRU Model...
Epoch [10/50], Train: 0.009709, Val: 0.034914
Epoch [20/50], Train: 0.004097, Val: 0.007187
Epoch [30/50], Train: 0.002636, Val: 0.004652
Epoch [40/50], Train: 0.002363, Val: 0.005168


2026-02-10 23:36:57,823 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


Epoch [50/50], Train: 0.002026, Val: 0.003683
Training improvement: 97.4%

Results for Junction 1:
GRU RMSE: 13.7017 | GRU+Fuzzy RMSE: 13.2231
GRU R²: 0.6949 | GRU+Fuzzy R²: 0.7158

Save model for Junction 1...
Model saved: models/gru_model_junction_1.pth
# Training Junction 2
------------------------------------------------------------

Junction 2
------------------------------------------------------------
Total records: 14592
After preprocessing: 14568 records
Sequences created: (14544, 24, 5)

PSO Optimization...


...

Epoch [50/50], Train: 0.002898, Val: 0.003651
Training improvement: 82.7%

Results for Junction 2:
GRU RMSE: 5.7992 | GRU+Fuzzy RMSE: 5.9487
GRU R²: 0.5493 | GRU+Fuzzy R²: 0.5258

Save model for Junction 2...
Model saved: models/gru_model_junction_2.pth
# Training Junction 3
------------------------------------------------------------

Junction 3
------------------------------------------------------------
Total records: 14592
After preprocessing: 14568 records
Sequences created: (14544, 24, 5)

PSO Optimization...


pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0.00294
2026-02-10 23:48:32,210 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0029391609132289886, best pos: [6.55267805e+01 1.10387792e+00 7.77985473e-03]


Best hidden_size: 65
Best num_layers: 1
Best learning_rate: 0.007780

Training Final GRU Model...
Epoch [10/50], Train: 0.002365, Val: 0.003332
Epoch [20/50], Train: 0.001291, Val: 0.002138
Epoch [30/50], Train: 0.001358, Val: 0.002260
Epoch [40/50], Train: 0.001155, Val: 0.002109
Epoch [50/50], Train: 0.001130, Val: 0.002028
Training improvement: 92.3%

Results for Junction 3:
GRU RMSE: 7.3165 | GRU+Fuzzy RMSE: 7.2986
GRU R²: 0.5492 | GRU+Fuzzy R²: 0.5514

Save model for Junction 3...
Model saved: models/gru_model_junction_3.pth
# Training Junction 4
------------------------------------------------------------

Junction 4
------------------------------------------------------------
Total records: 4344
After preprocessing: 4320 records
Sequences created: (4296, 24, 5)

PSO Optimization...


2026-02-10 23:49:11,686 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0.00512
2026-02-10 23:55:02,655 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.005120220128446817, best pos: [1.10281391e+02 2.40044147e+00 1.21473524e-03]


Best hidden_size: 110
Best num_layers: 2
Best learning_rate: 0.001215

Training Final GRU Model...
Epoch [10/50], Train: 0.006264, Val: 0.008316
Epoch [20/50], Train: 0.005736, Val: 0.007335
Epoch [30/50], Train: 0.005411, Val: 0.006910
Epoch [40/50], Train: 0.005280, Val: 0.006780
Epoch [50/50], Train: 0.005251, Val: 0.006698
Training improvement: 57.6%

Results for Junction 4:
GRU RMSE: 3.3686 | GRU+Fuzzy RMSE: 3.3398
GRU R²: 0.4139 | GRU+Fuzzy R²: 0.4239

Save model for Junction 4...
Model saved: models/gru_model_junction_4.pth
SUMMARY: ALL JUNCTIONS
--------------------------------------------------------------------------------

 Junction  RMSE_GRU  RMSE_Fuzzy   R2_GRU  R2_Fuzzy  Hidden_Size  Num_Layers  Learning_Rate
        1 13.701719   13.223085 0.694877  0.715822           56           1       0.006758
        2  5.799230    5.948664 0.549297  0.525770          112           1       0.007534
        3  7.316498    7.298556 0.549205  0.551413           65           1       0.0